# Tren YOLO-seg (Colab)

**Forberedelse (lokalt på PC):**
1. Høyreklikk `data`-mappen i Utforsker → **Komprimer til ZIP-fil** → kall den `data.zip`
2. Last opp `data.zip` til roten av Google Drive

**Kjør hele cellen under.**  
Modellen lagres automatisk til `trash-bin-models/` på Drive.

In [ ]:
import os
import sys
import shutil
import subprocess
import zipfile
from pathlib import Path
from datetime import datetime

# ============================================================
# INNSTILLINGER
# ============================================================

GITHUB_REPO_URL = "https://github.com/davgei/trash-bin-detection.git"
GITHUB_BRANCH = "streetview-yolo-retry"

REPO_DIR = Path("/content/trash-bin-detection")
MYDRIVE = Path("/content/drive/MyDrive")

ZIP_PATH = MYDRIVE / "data.zip"

EXTRACT_DIR = Path("/content/extracted_data")
DATA_DIR = REPO_DIR / "data" / "annotated_seg"

CONFIG_DIR = REPO_DIR / "configs"
DATA_YAML = CONFIG_DIR / "data_seg_colab.yaml"

RUNS_DIR = REPO_DIR / "runs"
RUN_NAME = "trash_bin_ground_yolo11n_seg"

MODEL_NAME = "yolo11n-seg.pt"
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 8
PATIENCE = 25

DRIVE_MODEL_DIR = MYDRIVE / "trash-bin-models"

CLASS_NAMES = {
    0: "trash_bin",
    1: "ground",
}


# ============================================================
# HJELPEFUNKSJONER
# ============================================================

def run(command, cwd=None):
    command = [str(x) for x in command]
    print("\n$", " ".join(command))
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Kommando feilet: {' '.join(command)}")
    return result


def count_files(folder, suffixes):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return sum(
        1 for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in suffixes
    )


def is_seg_label_file(label_path):
    try:
        lines = [
            line.strip()
            for line in label_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
    except Exception:
        return False
    if not lines:
        return False
    for line in lines:
        parts = line.split()
        if len(parts) >= 7:
            return True
    return False


def count_seg_labels(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*.txt") if is_seg_label_file(p))


def find_dataset_root(root):
    image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    candidates = []

    for images_train in root.rglob("images/train"):
        if not images_train.is_dir():
            continue
        candidate = images_train.parent.parent
        labels_train = candidate / "labels" / "train"
        if not labels_train.is_dir():
            continue

        train_images = count_files(candidate / "images" / "train", image_suffixes)
        val_images = count_files(candidate / "images" / "val", image_suffixes)
        test_images = count_files(candidate / "images" / "test", image_suffixes)
        train_seg_labels = count_seg_labels(candidate / "labels" / "train")
        val_seg_labels = count_seg_labels(candidate / "labels" / "val")
        test_seg_labels = count_seg_labels(candidate / "labels" / "test")

        total_images = train_images + val_images + test_images
        total_seg_labels = train_seg_labels + val_seg_labels + test_seg_labels
        is_annotated_seg = "annotated_seg" in candidate.parts

        if total_seg_labels == 0:
            score = 0
        else:
            score = total_seg_labels
            if is_annotated_seg:
                score += 100000

        candidates.append((score, candidate, total_images, total_seg_labels))

    if not candidates:
        raise RuntimeError(
            "Fant ikke YOLO-struktur i ZIP-en. "
            "Forventer images/train og labels/train."
        )

    candidates.sort(key=lambda x: x[0], reverse=True)
    print("\nMulige segmenteringsdatasett funnet:")
    for score, path, n_images, n_seg_labels in candidates:
        print(f"  {path}  score={score}, bilder={n_images}, seg-labels={n_seg_labels}")

    best_score, best_path, _, best_seg_labels = candidates[0]
    if best_score == 0 or best_seg_labels == 0:
        raise RuntimeError(
            "Fant ingen ekte YOLO-seg labels i ZIP-en. "
            "Label-filene ser ut som vanlige bokser, ikke polygoner."
        )
    return best_path


def safe_extract_zip(zip_path, dest):
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            member.filename = member.filename.replace("\\", "/")
            zf.extract(member, str(dest))


def verify_dataset(data_dir):
    image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    print("\nDatasettkontroll:")
    total_images = 0
    for split in ("train", "val", "test"):
        image_dir = data_dir / "images" / split
        label_dir = data_dir / "labels" / split
        n_images = count_files(image_dir, image_suffixes)
        n_labels = count_files(label_dir, {".txt"})
        total_images += n_images
        status = "OK" if n_images > 0 else "MANGLER"
        print(f"  {split:5s}: {n_images:4d} bilder, {n_labels:4d} labels [{status}]")
    if total_images == 0:
        raise RuntimeError("Ingen bilder funnet i datasettet.")
    if count_files(data_dir / "images" / "train", image_suffixes) == 0:
        raise RuntimeError("Mangler treningsbilder i images/train.")
    if count_files(data_dir / "labels" / "train", {".txt"}) == 0:
        raise RuntimeError("Mangler labels i labels/train.")


def write_data_yaml():
    CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    names_text = "\n".join(f"  {idx}: {name}" for idx, name in CLASS_NAMES.items())
    DATA_YAML.write_text(
        f"path: {DATA_DIR.as_posix()}\n\n"
        "train: images/train\n"
        "val: images/val\n"
        "test: images/test\n\n"
        f"names:\n{names_text}\n",
        encoding="utf-8",
    )
    print("\nLaget data yaml:")
    print(DATA_YAML)
    print(DATA_YAML.read_text())


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Fant ikke ZIP-filen: {ZIP_PATH}\n"
        "Last opp data.zip til roten av Google Drive."
    )

print(f"Fant ZIP: {ZIP_PATH}")
print(f"Størrelse: {ZIP_PATH.stat().st_size / 1024 / 1024:.1f} MB")


# ============================================================
# 2. KLON / OPPDATER REPO
# ============================================================

if not (REPO_DIR / ".git").exists():
    run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, REPO_DIR])
    print(f"Klonet branch: {GITHUB_BRANCH}")
else:
    run(["git", "-C", REPO_DIR, "pull"])
    print("Oppdaterte eksisterende repo.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


# ============================================================
# 3. PAKK UT ZIP OG FINN DATASETT
# ============================================================

local_zip = Path("/content/data.zip")
if local_zip.exists():
    local_zip.unlink()

print("\nKopierer ZIP til lokal Colab-disk...")
shutil.copy2(ZIP_PATH, local_zip)

print("Pakker ut ZIP...")
safe_extract_zip(local_zip, EXTRACT_DIR)
local_zip.unlink()

source_root = find_dataset_root(EXTRACT_DIR)
print(f"\nValgt datasettrot: {source_root}")

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(source_root / "images", DATA_DIR / "images")
shutil.copytree(source_root / "labels", DATA_DIR / "labels")
if (source_root / "previews").exists():
    shutil.copytree(source_root / "previews", DATA_DIR / "previews")
if (source_root / "review_flags.csv").exists():
    shutil.copy2(source_root / "review_flags.csv", DATA_DIR / "review_flags.csv")

print(f"\nData kopiert til: {DATA_DIR}")
verify_dataset(DATA_DIR)


# ============================================================
# 4. INSTALLER AVHENGIGHETER
# ============================================================

run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])


# ============================================================
# 5. LAG DATA YAML
# ============================================================

write_data_yaml()


# ============================================================
# 6. SJEKK GPU
# ============================================================

import torch

if torch.cuda.is_available():
    DEVICE = 0
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = "cpu"
    print("\nGPU ikke tilgjengelig. Trening går på CPU og kan ta lang tid.")


# ============================================================
# 7. TREN YOLO11 SEG
# ============================================================

from ultralytics import YOLO

training_output = RUNS_DIR / RUN_NAME
if training_output.exists():
    shutil.rmtree(training_output)

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
)


# ============================================================
# 8. LAGRE MODELL TIL DRIVE
# ============================================================

best_weight = training_output / "weights" / "best.pt"
last_weight = training_output / "weights" / "last.pt"

if not best_weight.exists():
    raise FileNotFoundError(f"Fant ikke best.pt etter trening: {best_weight}")

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(best_weight, DRIVE_MODEL_DIR / "yolo11n_seg_best.pt")
shutil.copy2(best_weight, DRIVE_MODEL_DIR / f"yolo11n_seg_best_{timestamp}.pt")
if last_weight.exists():
    shutil.copy2(last_weight, DRIVE_MODEL_DIR / "yolo11n_seg_last.pt")

print("\nFERDIG")
print(f"Beste modell lokalt:       {best_weight}")
print(f"Beste modell på Drive:     {DRIVE_MODEL_DIR / 'yolo11n_seg_best.pt'}")
print(f"Versjonert kopi på Drive:  {DRIVE_MODEL_DIR / f'yolo11n_seg_best_{timestamp}.pt'}")